# Laboratory 3: Deutsch's Algorithm
## Demonstrating Quantum Advantage

**Objective:** Implement the first quantum algorithm that proves quantum computers can outperform classical ones for a specific task.

**Learning Objectives:**
1. Understand the concept of a Quantum Oracle (Black Box).
2. Master the **Phase Kickback** mechanism.
3. Use quantum interference to determine global properties of a function in a single query.
4. *(Optional)* Execute the algorithm on a real quantum computer (QPU).

**The Problem:**
You are given a function $f: \{0,1\} \rightarrow \{0,1\}$. You want to know if it is **Constant** ($f(0)=f(1)$) or **Balanced** ($f(0) \neq f(1)$).
*   **Classical:** You need 2 queries (check $f(0)$ and $f(1)$).
*   **Quantum:** You need only **1 query**.

---

### Environment Setup

In [ ]:
# qiskit-ibm-runtime is pinned to the version tested with the optional QPU cell (October 2026)
!pip install qiskit[visualization] qiskit-aer "qiskit-ibm-runtime<0.51" matplotlib pylatexenc

import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_state_qsphere
from IPython.display import display

print("\n ENVIRONMENT SETUP COMPLETE!")

### Exercise 3: Deutsch's Algorithm and the Quantum Oracle

**Theoretical Context: Phase Kickback**
The secret of Deutsch's algorithm is preparing the ancilla (qubit 1) in the $|-\rangle$ state. 
When the Oracle applies $f(x)$, instead of changing the ancilla, the value of $f(x)$ is "kicked back" as a phase shift $(-1)^{f(x)}$ on the input qubit (qubit 0).

**Your Task:**
1. Run the cell as it is (`oracle_to_test = 'balanced'`: the CNOT oracle, $f(x)=x$). Expected: `{'1': 1000}`.
2. Change that line to `oracle_to_test = 'constant_0'` (no gate at all: $f(x)=0$) and run again. Expected: `{'0': 1000}`.
3. Change it to `'constant_1'` (an X on the ancilla: $f(x)=1$). Still constant, so expected `{'0': 1000}`.
4. *(Optional)* Add a fourth branch `elif oracle_type == 'balanced_not': qc.cx(0, 1); qc.x(1)` for $f(x)=1-x$, set `oracle_to_test = 'balanced_not'`, and check that it gives `{'1': 1000}`.

In [ ]:
def build_deutsch_circuit(oracle_type='balanced'):
    qc = QuantumCircuit(2, 1)

    # --- 1. STATE PREPARATION ---
    qc.x(1)           # Ancilla to |1>
    qc.h(0)           # Input to |+>
    qc.h(1)           # Ancilla to |->
    qc.barrier()

    # --- 2. THE ORACLE ---
    if oracle_type == 'balanced':
        qc.cx(0, 1)   # f(x) = x
    elif oracle_type == 'constant_0':
        pass          # f(x) = 0
    elif oracle_type == 'constant_1':
        qc.x(1)       # f(x) = 1

    qc.barrier()

    # --- 3. INTERFERENCE ---
    qc.h(0)           
    qc.measure(0, 0)  
    return qc

# Run the experiment
oracle_to_test = 'balanced' # Try 'constant_0' or 'constant_1'
qc = build_deutsch_circuit(oracle_to_test)

display(qc.draw('mpl'))
counts = AerSimulator().run(qc, shots=1000).result().get_counts()
print(f"Results for {oracle_to_test} oracle: {counts}")
if '1' in counts: print("Conclusion: BALANCED")
else: print("Conclusion: CONSTANT")
plot_histogram(counts)

### OPTIONAL: Execute on a Real Quantum Computer (QPU)

Now we send the *same* circuit to a real IBM chip. 
**Note:** Real chips have noise. You won't get a perfect 100%, but you will see the "Quantum Signal" clearly above the noise floor.

**How to get access** (free; about ten minutes the first time):

1. Sign in at [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) with an IBMid (you can create one there).
2. If the platform says that an IBM Cloud account is required, follow its **Create an IBM Cloud account** link.
3. On the **Instances** page, create an instance with the **Open** plan (region us-east).
4. On the platform's home dashboard, create an **API key** and copy it.

Treat the key like a password: paste it only into `API_KEY` in the cell below, never share a notebook that
contains it, and delete the key on the dashboard if it leaks. The Open Plan gives 10 minutes of QPU time every
28 days (October 2026); one run of the cell below uses well under a minute of it. Everything else in this lab
runs on the simulator and needs no account.

In [ ]:
# OPTIONAL: run the same circuit on a real IBM quantum computer (see "How to get access" above).
import warnings
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# The version of qiskit-ibm-runtime installed above (tested in October 2026) marks SamplerV2 as deprecated;
# it still works, so that warning is hidden.
warnings.filterwarnings("ignore", message=".*SamplerV2 class is deprecated.*")

API_KEY = ""   # paste your IBM Quantum Platform API key between the quotes (never share it)


def run_on_hardware(circuit, api_key):
    """Send the circuit to the least busy IBM QPU and return the job (None if no key is set)."""
    if not api_key.strip():
        print("No API key: nothing was sent. Paste your key into API_KEY and run this cell again.")
        return None
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=api_key.strip())
    backend = service.least_busy(operational=True, simulator=False)
    print(f"Selected backend: {backend.name}")
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
    job = SamplerV2(mode=backend).run([pm.run(circuit)])
    print(f"Job ID: {job.job_id()} (it may wait in a queue for a few minutes)")
    return job


job = run_on_hardware(qc, API_KEY)
if job is not None:
    counts_hw = job.result()[0].data.c.get_counts()   # waits until the job has run
    print(f"Hardware counts: {counts_hw}")
    display(plot_histogram(counts_hw))